# DunedinPoAm38

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.DunedinPoAm38)

class DunedinPoAm38(LinearReferenceClock):
    pass



In [3]:
model = pya.models.DunedinPoAm38()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "dunedinpoam38"
model.metadata["data_type"] = "DNA methylation"  # Paper: DunedinPoAm uses blood DNA methylation.
model.metadata["species"] = "Homo sapiens"  # Paper: The model was developed in members of the human Dunedin Study.
model.metadata["year"] = 2020
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Belsky, D.W., Caspi, A., Arseneault, L. et al. Quantification of the pace of biological aging in humans through a blood test, the DunedinPoAm DNA methylation algorithm. eLife 9, e54870 (2020)."
model.metadata["doi"] = "https://doi.org/10.7554/elife.54870"
model.metadata["notes"] = "Whole-blood elastic-net estimator of the rate of biological aging, trained at age 38 against a longitudinal 18-biomarker Pace-of-Aging composite measured over ages 26–38."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: DNA methylation at age 38 was measured in whole blood.
model.metadata["predicts"] = ["pace of aging"]  # Paper: DunedinPoAm is a rate measure of how fast biological aging is occurring.
model.metadata["training_target"] = ["pace of aging"]  # Paper: Elastic net was fitted to the Pace-of-Aging composite derived from 18 biomarkers at ages 26, 32 and 38.
model.metadata["unit"] = ["biological years per chronological year"]  # Paper: A value near 1 represents one year of biological aging per calendar year.
model.metadata["model_type"] = "elastic net regression"  # Paper: Elastic-net regression was used to derive the methylation algorithm.
model.metadata["platform"] = ["Illumina 450K"]  # Paper: Feature selection and model fitting used age-38 Dunedin whole-blood methylation measured on the 450K array; EPIC was used for later external application.
model.metadata["population"] = "adults"  # Paper: The target was developed in 954 members of the same-year Dunedin birth cohort.
model.metadata["journal"] = "eLife"
model.metadata["last_author"] = "Terrie E. Moffitt"
model.metadata["n_features"] = 46
model.metadata["citations"] = 666
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
os.system(f"curl -sL -o coefficients.csv https://raw.githubusercontent.com/bio-learn/biolearn/180852e2bab473303cb85da627178b1695ee9d86/biolearn/data/DunedinPoAm38.csv")

0

## Load features

In [6]:
df = pd.read_csv('coefficients.csv')
mask = df['CpGmarker'].astype(str).str.lower().isin(['intercept', '(intercept)'])
intercept_value = float(df.loc[mask, 'CoefficientTraining'].iloc[0]) if mask.any() else 0.0
coef_df = df.loc[~mask].reset_index(drop=True)
model.features = coef_df['CpGmarker'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['CoefficientTraining'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([intercept_value]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Belsky, Daniel W., et al. "Quantification of the pace of '
             'biological aging in humans through a blood test, the DunedinPoAm '
             'DNA methylation algorithm." eLife 9 (2020): e54870.',
 'clock_name': 'dunedinpoam38',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.7554/eLife.54870',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2020}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg02582848', 'cg03730474', 'cg03922834', 'cg04480708', 'cg05227215', 'cg05513157', 'cg05575921', 'cg06133392', 'cg07045089', 'cg07185119', 'cg07986378', 'cg08376310', 'cg09349128', 'cg09404119', 'cg10727171', 'cg10919522', 'cg11574055', 'cg11674508', 'cg11897887', 

## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: coefficients.csv
